In [ ]:
! pip install kaggle


Downloaded the Dataset

In [ ]:
!kaggle datasets download kazanova/sentiment140

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
100% 80.9M/80.9M [00:00<00:00, 99.3MB/s]



Extraction the csv file from the zip

In [ ]:
from zipfile import ZipFile
dataset = '/content/sentiment140.zip'

with ZipFile(dataset,'r') as zip:
  zip.extractall()
  print('Extraction Done')

Extraction Done


Importing the required dependencies

In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

Loading the data


In [ ]:
twitter_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv' , encoding = 'ISO-8859-1')

In [ ]:
coloumn_names = ['Target',  'ID' , 'Date' , 'Flag' , 'User' , 'Text']
twitter_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv' ,names = coloumn_names, encoding = 'ISO-8859-1')

In [ ]:
twitter_data.shape

(1600000, 6)

In [ ]:
twitter_data.head()

,Target,ID,Date,Flag,User,Text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [ ]:
twitter_data.isnull().sum()

,0
Target,0
ID,0
Date,0
Flag,0
User,0
Text,0


In [ ]:
twitter_data['Target'].value_counts()

,count
Target,
0,800000
4,800000


Converting the target 4 -> 1

Now onwards -

1 --> positive



0 --> negative

In [ ]:
twitter_data.replace({'Target':{4:1}},inplace = True)

In [ ]:
twitter_data['Target'].value_counts()

,count
Target,
0,800000
1,800000


In [ ]:
port_stem = PorterStemmer()

In [ ]:
def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]',' ',content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)

  return stemmed_content

In [ ]:
twitter_data['Stemmed_content'] = twitter_data['Text'].apply(stemming)

KeyboardInterrupt: 

In [ ]:
twitter_data.head()

In [ ]:
x = twitter_data['Stemmed_content'].values
y = twitter_data['Target'].values

In [ ]:
print(x)

In [ ]:
print(y)

In [ ]:
x_train , x_test , y_train , y_test = train_test_split(x,y,test_size = 0.2,stratify = y,random_state = 2)

In [ ]:
print(x_train)

In [ ]:
print(x_test)

In [ ]:
vectorizer = TfidfVectorizer()
x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)

In [ ]:
import pickle

In [ ]:
import os
os.makedirs('project', exist_ok=True)

In [ ]:
pickle.dump(vectorizer, open('project/vectorizer.sav', 'wb'))

In [ ]:
print(x_train)

In [ ]:
print(x_test)

Now training the Machine Learning Model

In [ ]:
model = LogisticRegression(max_iter = 1000)

In [ ]:
model.fit(x_train,y_train)

Now doing model Evaluation

In [ ]:
x_train_prediction = model.predict(x_train)
training_data_accuracy = accuracy_score(y_train,x_train_prediction)

In [ ]:
print("Accuracy Score : " , training_data_accuracy)

In [ ]:
x_test_prediction = model.predict(x_test)
test_data_accuracy = accuracy_score(y_test,x_test_prediction)

In [ ]:
print("Accuracy Score : " , test_data_accuracy)

Saving the Trained Model

In [ ]:
import pickle

In [ ]:
filename = "trained_model.sav"
pickle.dump(model,open(filename,'wb'))

Now using the saved model for Future Predictions

In [ ]:
loaded_model = pickle.load(open('trained_model.sav','rb'))


In [ ]:
x_new = x_test[200]
print(y_test[200])

prediction = loaded_model.predict(x_new)
print(prediction)

if prediction[0]=='0':
  print("Negative Tweet")
else:
  print("Positive Tweet")

In [ ]:
x_new = x_test[5]
print(y_test[5])

prediction = loaded_model.predict(x_new)
print(prediction)

if prediction[0]=='0':
  print("Negative Tweet")
else:
  print("Positive Tweet")

---
---

# Part 2 — Delete, Retrieve, Generate (Li et al., NAACL 2018)

Everything above is **unchanged**. This section only *adds* the artifacts needed for
negative → positive sentiment style transfer. Nothing here overwrites `trained_model.sav`
or `vectorizer.sav`, and nothing above depends on anything below.

**Why this fits what we already built.** In a TF-IDF + LogisticRegression model,
`model.coef_` is literally a sentiment score per n-gram. DRG's hardest conceptual step —
*"which words carry the sentiment?"* — collapses into a coefficient lookup. No GPU, no
new architecture.

**The one thing we cannot reuse is `Stemmed_content`.** `stemming()` throws away casing,
punctuation and word endings, so `"disappointed"` → `"disappoint"` can never be turned back
into a fluent tweet. So we build a **second** vectorizer + LogReg on lightly-cleaned,
*unstemmed* text, used **only to mine markers**. Bigrams matter here (`not good`, `waste of`),
hence `ngram_range=(1, 2)`.

The original classifier stays exactly as it is, and gets reused in the next notebook as the
**evaluator** for the style transfer.

### Part 2 config

In [ ]:
import os
import re
import json
import pickle
import shutil

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression

MARKER_SAMPLE_SIZE    = 200000   # tweets used to mine attribute markers (bump if you have RAM)
RETRIEVAL_CORPUS_SIZE = 30000    # positive tweets kept as the retrieval index
TOP_K_MARKERS         = 1500     # markers kept per polarity
MIN_DF                = 20       # an n-gram must appear in >= this many tweets
SALIENCE_THRESHOLD    = 1.5      # Li et al.'s salience ratio, used as a precision guard
EVAL_SET_SIZE         = 300      # held-out negative tweets saved for notebook 2's evaluation

# Plain \b\w+\b so that the vectorizer's tokens are EXACTLY a whitespace split of our
# cleaned text. sklearn's default (\b\w\w+\b) drops 1-char tokens and would silently
# desync the DELETE step from the vocabulary.
TOKEN_PATTERN = r"(?u)\b\w+\b"

os.makedirs('project', exist_ok=True)

### Light cleaning (the reversible alternative to `stemming()`)

Apostrophes are removed rather than replaced with a space (`don't` → `dont`) so that a plain
`.split()` gives exactly the tokens the vectorizer indexes. That single detail is what keeps
the DELETE step and the marker vocabulary in sync.

In [ ]:
def light_clean(content):
    """Human-readable cleaning. Unlike stemming(), the output can still be generated from."""
    content = content.lower()
    content = re.sub(r'http\S+|www\.\S+', ' ', content)   # urls
    content = re.sub(r'@\w+', ' ', content)               # @mentions
    content = re.sub(r"'", '', content)                   # don't -> dont
    content = re.sub(r'[^a-z\s]', ' ', content)           # letters only
    content = re.sub(r'\s+', ' ', content).strip()
    return content


print(light_clean("I'm SO disappointed with @apple ... battery is terrible!! http://t.co/x1"))

In [ ]:
twitter_data['Clean_text'] = twitter_data['Text'].apply(light_clean)
twitter_data[['Text', 'Stemmed_content', 'Clean_text']].head()

In [ ]:
# Drop 1-3 word fragments: nothing to preserve, nothing to transfer.
drg_data = twitter_data[twitter_data['Clean_text'].str.split().str.len() >= 4]

drg_sample = drg_data.sample(n=min(MARKER_SAMPLE_SIZE, len(drg_data)), random_state=2)

print('usable tweets :', drg_data.shape[0])
print('marker sample :', drg_sample.shape[0])
print(drg_sample['Target'].value_counts())

### Mining attribute markers

Two signals, combined:

1. **LogReg coefficient** — how much an n-gram pushes the decision toward positive / negative.
2. **Salience ratio** (Li et al., §3.1) — `(df in target polarity + λ) / (df in other polarity + λ)`.
   This is the guard: it kills n-grams that got a big coefficient from a handful of tweets.

In [ ]:
marker_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=MIN_DF,
    max_features=50000,
    token_pattern=TOKEN_PATTERN,
)
X_marker = marker_vectorizer.fit_transform(drg_sample['Clean_text'].values)
y_marker = drg_sample['Target'].values

marker_model = LogisticRegression(max_iter=1000)
marker_model.fit(X_marker, y_marker)

print('vocabulary size          :', len(marker_vectorizer.vocabulary_))
print('marker model train acc   :', round(marker_model.score(X_marker, y_marker), 4))

In [ ]:
# Document frequency of every vocab n-gram, per polarity -> salience ratio.
count_vectorizer = CountVectorizer(
    vocabulary=marker_vectorizer.vocabulary_,
    ngram_range=(1, 2),
    token_pattern=TOKEN_PATTERN,
    binary=True,
)
C = count_vectorizer.transform(drg_sample['Clean_text'].values)

pos_df = np.asarray(C[y_marker == 1].sum(axis=0)).ravel()
neg_df = np.asarray(C[y_marker == 0].sum(axis=0)).ravel()

lam = 1.0
pos_salience = (pos_df + lam) / (neg_df + lam)
neg_salience = (neg_df + lam) / (pos_df + lam)

print('salience computed for', len(pos_salience), 'n-grams')

In [ ]:
feature_names = np.array(marker_vectorizer.get_feature_names_out())
coefs = marker_model.coef_[0]
order = np.argsort(coefs)                      # most negative -> most positive

# take a generous slice by coefficient, then keep only what also passes salience
neg_idx = [i for i in order[:TOP_K_MARKERS * 3]
           if neg_salience[i] >= SALIENCE_THRESHOLD][:TOP_K_MARKERS]
pos_idx = [i for i in order[::-1][:TOP_K_MARKERS * 3]
           if pos_salience[i] >= SALIENCE_THRESHOLD][:TOP_K_MARKERS]

neg_markers = set(feature_names[neg_idx])
pos_markers = set(feature_names[pos_idx])

print(f'negative markers : {len(neg_markers)}')
print(f'positive markers : {len(pos_markers)}')
print()
print('top 30 negative  :', list(feature_names[neg_idx][:30]))
print()
print('top 30 positive  :', list(feature_names[pos_idx][:30]))

### DELETE

Bigrams are matched greedily first (`not good` must beat `good`), then leftover unigrams.
O(sentence length) with set lookups — not O(number of markers) — so this stays fast over 30k+ tweets.

In [ ]:
def delete_markers(sentence, marker_set):
    """DELETE step. Strips one polarity's attribute markers.
    Returns (content_template, [markers that were removed])."""
    words = sentence.split()
    n = len(words)
    drop = [False] * n
    found = []

    i = 0
    while i < n - 1:                                   # bigrams first
        bg = words[i] + ' ' + words[i + 1]
        if bg in marker_set:
            drop[i] = drop[i + 1] = True
            found.append(bg)
            i += 2
        else:
            i += 1

    for i in range(n):                                 # then leftover unigrams
        if not drop[i] and words[i] in marker_set:
            drop[i] = True
            found.append(words[i])

    template = ' '.join(w for i, w in enumerate(words) if not drop[i])
    return template, found


demo = light_clean("the battery on this phone is terrible and the support team is useless")
print('clean    :', demo)
print('template :', delete_markers(demo, neg_markers)[0])
print('deleted  :', delete_markers(demo, neg_markers)[1])

### RETRIEVE

Index every positive tweet by its *content template* (i.e. with its positive markers stripped).
Then, given a negative tweet's template, the nearest neighbour in that index is a positive tweet
**about the same thing** — and we borrow the positive markers it was using.

In [ ]:
pos_pool = drg_data[drg_data['Target'] == 1].sample(
    n=min(RETRIEVAL_CORPUS_SIZE, int((drg_data['Target'] == 1).sum())),
    random_state=2,
)

pos_corpus, pos_templates, pos_template_markers = [], [], []
for s in pos_pool['Clean_text'].tolist():
    t, m = delete_markers(s, pos_markers)
    if m and len(t.split()) >= 2:          # must have markers to lend AND content to match on
        pos_corpus.append(s)
        pos_templates.append(t)
        pos_template_markers.append(m)

retrieval_vectorizer = TfidfVectorizer(token_pattern=TOKEN_PATTERN)
pos_template_matrix = retrieval_vectorizer.fit_transform(pos_templates)

print('retrieval index :', pos_template_matrix.shape)
print('example ->', pos_corpus[0])
print('  template :', pos_templates[0])
print('  markers  :', pos_template_markers[0])

In [ ]:
def retrieve(template, top_n=1):
    """RETRIEVE step. Nearest positive tweet by CONTENT; hand back its positive markers."""
    q = retrieval_vectorizer.transform([template])
    sims = (pos_template_matrix @ q.T).toarray().ravel()   # tf-idf is l2-normalised => cosine
    idx = np.argsort(sims)[::-1][:top_n]
    return [{'markers':   pos_template_markers[i],
             'neighbour': pos_corpus[i],
             'sim':       float(sims[i])} for i in idx]

### GENERATE — the model-free baseline

`TemplateBased` in the paper: splice the borrowed positive markers back into the template.
It is meant to look clumsy. That clumsiness is exactly the argument for putting an LLM in the
GENERATE slot in notebook 2 — and it gives you a real baseline row for your results table.

In [ ]:
def delete_only(tweet):
    """DRG's DeleteOnly variant: strip negative markers, add nothing back."""
    clean = light_clean(tweet)
    template, deleted = delete_markers(clean, neg_markers)
    return template


def template_based(tweet):
    """DRG's TemplateBased variant: delete negative markers, splice in borrowed positive ones."""
    clean = light_clean(tweet)
    template, deleted = delete_markers(clean, neg_markers)
    hits = retrieve(template)
    borrowed = hits[0]['markers'][:2] if hits else []
    return {
        'input':     tweet,
        'clean':     clean,
        'template':  template,
        'deleted':   deleted,
        'borrowed':  borrowed,
        'neighbour': hits[0]['neighbour'] if hits else '',
        'sim':       hits[0]['sim'] if hits else 0.0,
        'output':    (' '.join(borrowed) + ' ' + template).strip(),
    }


for t in [
    "the battery on this phone is terrible and the support team is useless",
    "worst customer service ever, waited 2 hours and nobody even helped me",
    "this update broke everything, so frustrated right now",
]:
    d = template_based(t)
    print('IN   :', d['input'])
    print('DEL  :', d['deleted'])
    print('TPL  :', d['template'])
    print('BOR  :', d['borrowed'], f"(sim={d['sim']:.2f}  <- {d['neighbour'][:60]})")
    print('OUT  :', d['output'])
    print()

### Held-out evaluation set

300 negative tweets that were **not** in the marker sample. Notebook 2 uses these to report
style accuracy / content preservation, so the numbers aren't measured on tweets the markers
were mined from.

In [ ]:
eval_pool = drg_data[(drg_data['Target'] == 0) & (~drg_data.index.isin(drg_sample.index))]
eval_negatives = eval_pool.sample(n=min(EVAL_SET_SIZE, len(eval_pool)), random_state=99)['Text'].tolist()

with open('project/eval_negatives.json', 'w') as f:
    json.dump(eval_negatives, f)

print(len(eval_negatives), 'held-out negative tweets saved')
for t in eval_negatives[:3]:
    print(' -', t)

### Save the bundle

`project/` ends up self-contained: the original classifier (for the app and for evaluation),
plus everything DRG needs at inference time.

In [ ]:
# markers -> json (small, human-readable, you can hand-edit it if a marker looks wrong)
with open('project/drg_markers.json', 'w') as f:
    json.dump({'neg_markers': sorted(neg_markers),
               'pos_markers': sorted(pos_markers)}, f)

# retrieval index -> pickle (fitted vectorizer + sparse matrix + corpus)
with open('project/drg_retrieval.sav', 'wb') as f:
    pickle.dump({'retrieval_vectorizer': retrieval_vectorizer,
                 'pos_template_matrix':  pos_template_matrix,
                 'pos_templates':        pos_templates,
                 'pos_template_markers': pos_template_markers,
                 'pos_corpus':           pos_corpus}, f)

# the marker model itself -- not needed at inference, kept for the report
pickle.dump(marker_model,      open('project/marker_model.sav', 'wb'))
pickle.dump(marker_vectorizer, open('project/marker_vectorizer.sav', 'wb'))

# the ORIGINAL classifier, so project/ is one self-contained folder for the app
pickle.dump(model,      open('project/trained_model.sav', 'wb'))
pickle.dump(vectorizer, open('project/vectorizer.sav', 'wb'))

for fn in sorted(os.listdir('project')):
    print(f"{fn:26s} {os.path.getsize(os.path.join('project', fn)) / 1e6:8.2f} MB")

In [ ]:
shutil.make_archive('project_bundle', 'zip', 'project')
print('project_bundle.zip :', round(os.path.getsize('project_bundle.zip') / 1e6, 2), 'MB')

try:
    from google.colab import files
    files.download('project_bundle.zip')
except Exception:
    print('(not on Colab -- grab project_bundle.zip from the file browser)')

### What you now have

| file | used by |
|---|---|
| `trained_model.sav`, `vectorizer.sav` | Streamlit app (classify) + notebook 2 (evaluate) |
| `drg_markers.json` | DELETE step |
| `drg_retrieval.sav` | RETRIEVE step |
| `eval_negatives.json` | notebook 2's results table |
| `marker_model.sav`, `marker_vectorizer.sav` | report only |

Next: upload `project_bundle.zip` to **`Style_Transfer_DRG.ipynb`**.

If the top markers printed above look noisy, the two dials are `MIN_DF` (raise it) and
`SALIENCE_THRESHOLD` (raise it). You can also just delete bad entries straight out of
`drg_markers.json` — it's plain text on purpose.